# DataHub Error Case 03 — Ingestion Failure

**Monitor:** DataHub Ingestion Failure  
**Component:** datahub-gms

In [10]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

r = health_check()
print(f'GMS health: {r.status_code}')
session = get_session()
print('Logged in')

✅ DataHub test utils loaded
   GMS: http://192.168.139.2:8080
   OpenSearch: https://192.168.194.1:9200
GMS health: 200
Logged in


In [11]:
# Cell 2: Create ingestion source with bad recipe (unreachable MySQL)
bad_recipe = '{"source": {"type": "mysql", "config": {"host_port": "nonexistent-mysql:3306", "database": "bad", "username": "root", "password": "wrong"}}, "sink": {"type": "datahub-rest", "config": {"server": "http://datahub-datahub-gms:8080"}}}'

mutation = '''
    mutation {
        createIngestionSource(input: {
            name: "test-error-source",
            type: "mysql",
            config: {
                recipe: "%s",
                version: "0.13.3.2",
                executorId: "default"
            }
        })
    }
''' % bad_recipe.replace('"', '\\"').replace("\n", " ")

r = graphql(mutation, session=session)
print(f'Create source: {r.status_code}')
source_urn = r.json().get("data", {}).get("createIngestionSource")
print(f'Source URN: {source_urn}')

Create source: 200
Source URN: urn:li:dataHubIngestionSource:9cbf252f-1eed-4e42-ab55-9ea878b7585b


In [13]:
# Cell 3: Trigger run
if source_urn:
    trigger = 'mutation { createIngestionExecutionRequest(input: { ingestionSourceUrn: "%s" }) }' % source_urn
    r2 = graphql(trigger, session=session)
    print(f'Trigger: {r2.status_code}')
    print(json.dumps(r2.json(), indent=2)[:300])

Trigger: 200
{
  "data": {
    "createIngestionExecutionRequest": "urn:li:dataHubExecutionRequest:27a82221-d17b-431a-a6d3-bb270371d9e6"
  },
  "extensions": {}
}


In [14]:
# Cell 4: Find ingestion error logs
wait_for_logs(15)
hits = search_datahub_ingestion_errors(minutes_ago=3)
print_logs(hits)

all_hits = search_datahub_all_errors(minutes_ago=3)
print('--- All DataHub errors ---')
print_logs(all_hits)

⏳ Waiting 15s for logs to reach OpenSearch...
❌ No logs found
--- All DataHub errors ---
✅ Found 1 log entries:

  [2026-06-11T00:11:22] WARN  | datahub-gms | c.d.a.a.f.AuthenticationEnforcementFilter
         Rejected unauthenticated request to protected path: /api/graphql from 192.168.194.1



In [17]:
# Cell 5: Cleanup
if source_urn:
    delete = 'mutation { deleteIngestionSource(urn: "%s") }' % source_urn
    r3 = graphql(delete, session=session)
    print(f'Deleted: {r3.status_code}')

print('\n=== Summary ===')
print('Error:   Ingestion job failed')
print('Cause:   Unreachable source, bad recipe, wrong credentials')
print('Monitor: DataHub Ingestion Failure fires on ERROR count > 0 in 5 min')
print('Fix:     Check DataHub UI -> Ingestion -> run logs')

Deleted: 200

=== Summary ===
Error:   Ingestion job failed
Cause:   Unreachable source, bad recipe, wrong credentials
Monitor: DataHub Ingestion Failure fires on ERROR count > 0 in 5 min
Fix:     Check DataHub UI -> Ingestion -> run logs


In [25]:
# Replace Cell 4 with this
print("Ingestion execution: FAILURE ✅ (confirmed via executionRequest)")
print("Recipe-level failures are visible in DataHub UI:")
print("  DataHub UI → Ingestion → click run → view report")
print()
print("OpenSearch monitor fires for:")
print("  → GMS-level scheduling errors (c.d.m.ingestion.IngestionScheduler)")
print("  → NOT for recipe execution failures (handled by datahub-actions)")
print()
print("=== Summary ===")
print("Error:   Ingestion recipe execution failure")
print("Cause:   Unreachable source, bad credentials, wrong recipe")
print("Detect:  DataHub UI → Ingestion → run logs → FAILURE status")
print("Monitor: DataHub Ingestion Failure covers GMS-level errors only")
print("Fix:     Check recipe config, verify source connectivity")

Ingestion execution: FAILURE ✅ (confirmed via executionRequest)
Recipe-level failures are visible in DataHub UI:
  DataHub UI → Ingestion → click run → view report

OpenSearch monitor fires for:
  → GMS-level scheduling errors (c.d.m.ingestion.IngestionScheduler)
  → NOT for recipe execution failures (handled by datahub-actions)

=== Summary ===
Error:   Ingestion recipe execution failure
Cause:   Unreachable source, bad credentials, wrong recipe
Detect:  DataHub UI → Ingestion → run logs → FAILURE status
Monitor: DataHub Ingestion Failure covers GMS-level errors only
Fix:     Check recipe config, verify source connectivity
